<a href="https://colab.research.google.com/github/VHARSHILJOSEPH/MileStone2-Infosis-/blob/main/SpringBoot(milestone2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📦 Step 1: Install dependencies
!pip install spacy py2neo openai networkx matplotlib
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ⚙️ Step 2: Import libraries
import spacy
import openai
from py2neo import Graph, Node, Relationship
import matplotlib.pyplot as plt
import networkx as nx
import json
from IPython.display import display

In [ ]:
# 🔐 Step 3: Set API keys and Neo4j credentials
openai.api_key = "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"  # Replace with your OpenAI API key

# Replace with your Neo4j Aura Free credentials or local Neo4j
neo4j_url = "https://console.neo4j.io/org/90856581-4131-4643-ba58-cb93821a0aa2/projects"
neo4j_user = "5c38a1b8"
neo4j_password = "JPj6oMEB4DPyNviTzwrUEDILLKBYQQ4f_IIQkOn-9xM"

graph = Graph(neo4j_url, auth=(neo4j_user, neo4j_password))

ConnectionUnavailable: ('Cannot open connection to %r', ConnectionProfile('https://console.neo4j.io:7473'))

In [ ]:
# 📄 Step 4: Input your document
text = """
OpenAI was founded by Sam Altman and Elon Musk.
It is headquartered in San Francisco, California.
In 2023, OpenAI released GPT-4.
"""

In [ ]:
# 🤖 Step 5: Run spaCy NER
nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

entities = []
for ent in doc.ents:
    entities.append({"text": ent.text, "label": ent.label_})

entities

In [ ]:
# 🧠 Step 6: Use OpenAI to extract relations
prompt = f"""
Extract entities and their relationships from the text. Output only valid JSON:

Text:
\"\"\"{text}\"\"\"

Output JSON format:
{{
  "entities": [{{"id": "E1", "text": "Sam Altman", "type": "PERSON"}}],
  "relations": [
    {{"source_id": "E1", "target_id": "E2", "label": "FOUNDED", "confidence": 0.95}}
  ]
}}
"""

response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_tokens=500,
)

output_json = json.loads(response["choices"][0]["message"]["content"])
display(output_json)

In [ ]:
# 🔁 Step 7: Build graph in Neo4j
graph.run("MATCH (n) DETACH DELETE n")  # (Optional) clear graph

# Insert entities
entity_map = {}
for ent in output_json["entities"]:
    node = Node("Entity", id=ent["id"], name=ent["text"], type=ent["type"])
    graph.merge(node, "Entity", "id")
    entity_map[ent["id"]] = node

# Insert relationships
for rel in output_json["relations"]:
    source = entity_map.get(rel["source_id"])
    target = entity_map.get(rel["target_id"])
    if source and target:
        r = Relationship(source, rel["label"], target, confidence=rel.get("confidence", 1.0))
        graph.merge(r)

In [ ]:
# 🔎 Step 8: Visualize graph in Colab
G = nx.DiGraph()

for ent in output_json["entities"]:
    G.add_node(ent["id"], label=ent["text"], type=ent["type"])

for rel in output_json["relations"]:
    G.add_edge(rel["source_id"], rel["target_id"], label=rel["label"])

plt.figure(figsize=(8,6))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=False, node_color='skyblue', node_size=2000, arrows=True)
nx.draw_networkx_labels(G, pos, labels={n:d['label'] for n,d in G.nodes(data=True)})
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u,v):d['label'] for u,v,d in G.edges(data=True)})
plt.title("Entity Relationship Graph")
plt.axis('off')
plt.show()